# 🏟️ FutPythonTrader API — Test Notebook

Notebook para verificar e explorar todos os endpoints da API `https://api.futpythontrader.com`.

| Endpoint | Método | Descrição |
|---|---|---|
| `/api/dados/` | GET | Lista fontes disponíveis |
| `/api/dados/jogos-do-dia/{fonte}/datas/` | GET | Datas disponíveis |
| `/api/dados/jogos-do-dia/{fonte}/{data}/` | GET | Jogos do dia por fonte e data |
| `/api/dados/jogos-do-dia/{fonte}/{data}/download/` | GET | Download CSV de jogos do dia |
| `/api/dados/{fonte}/download/` | GET | Download CSV base histórica completa |

## 0. Setup & Autenticação

In [ ]:
import requests
import pandas as pd
import io
import json
from datetime import date, timedelta
from IPython.display import display

# ── Autenticação ───────────────────────────────────────────────────────────────
BASE_URL = "https://api.futpythontrader.com"
TOKEN    = "95baedae25e7bd8811a6ad4a505bae6210bf3dcc"
HEADERS  = {"Authorization": f"Token {TOKEN}"}

# ── Helpers ────────────────────────────────────────────────────────────────────
SOURCES  = ["bet365", "betfair", "footystats"]
HOJE     = date.today().isoformat()
ONTEM    = (date.today() - timedelta(days=1)).isoformat()

def pp(data, max_rows=5):
    """Pretty-print: dict → json, list[dict] → DataFrame."""
    if isinstance(data, list) and data and isinstance(data[0], dict):
        return display(pd.DataFrame(data).head(max_rows))
    print(json.dumps(data, indent=2, ensure_ascii=False))

print(f"✅ Setup OK | hoje={HOJE} | ontem={ONTEM}")
print(f"   HEADERS: {HEADERS}")

## 1. Listar Fontes — GET `/api/dados/`

In [ ]:
r = requests.get(f"{BASE_URL}/api/dados/", headers=HEADERS, timeout=15)
print(f"Status: {r.status_code}")

if r.status_code == 200:
    pp(r.json())
else:
    print(f"❌ {r.text}")

## 2. Datas disponíveis — GET `/api/dados/jogos-do-dia/{fonte}/datas/`

In [ ]:
for fonte in SOURCES:
    url = f"{BASE_URL}/api/dados/jogos-do-dia/{fonte}/datas/"
    r = requests.get(url, headers=HEADERS, timeout=15)
    print(f"\n{'─'*60}")
    print(f"📅 {fonte.upper()} — Status: {r.status_code}")

    if r.status_code == 200:
        datas = r.json()
        datas_list = datas if isinstance(datas, list) else datas.get("datas", [])
        print(f"   Total datas: {len(datas_list)}")
        print(f"   Últimas 5:   {sorted(datas_list)[-5:]}")
    else:
        print(f"   ❌ {r.text[:300]}")

## 3. Jogos do Dia — GET `/api/dados/jogos-do-dia/{fonte}/{data}/`

In [ ]:
# ── Bet365 — hoje e ontem ──────────────────────────────────────────────────────
fonte = "bet365"

for DIA in [HOJE, ONTEM]:
    url = f"{BASE_URL}/api/dados/jogos-do-dia/{fonte}/{DIA}/"
    r = requests.get(url, headers=HEADERS, timeout=15)
    print(f"\n📆 {fonte.upper()} | {DIA} — Status: {r.status_code} | URL: {url}")

    if r.status_code == 200:
        payload = r.json()
        dados = payload if isinstance(payload, list) else payload.get("dados", [])
        df = pd.DataFrame(dados)
        print(f"   ✅ {len(df)} jogos | Colunas: {list(df.columns)}")
        display(df.head())
    else:
        print(f"   ⚠️  {r.text[:300]}")

In [ ]:
# ── Betfair ───────────────────────────────────────────────────────────────────
fonte = "betfair"

for DIA in [HOJE, ONTEM]:
    url = f"{BASE_URL}/api/dados/jogos-do-dia/{fonte}/{DIA}/"
    r = requests.get(url, headers=HEADERS, timeout=15)
    print(f"\n📆 {fonte.upper()} | {DIA} — Status: {r.status_code}")

    if r.status_code == 200:
        payload = r.json()
        dados = payload if isinstance(payload, list) else payload.get("dados", [])
        df = pd.DataFrame(dados)
        print(f"   ✅ {len(df)} jogos | Colunas: {list(df.columns)}")
        display(df.head())
    else:
        print(f"   ⚠️  {r.text[:300]}")

In [ ]:
# ── FootyStats ────────────────────────────────────────────────────────────────
fonte = "footystats"

for DIA in [HOJE, ONTEM]:
    url = f"{BASE_URL}/api/dados/jogos-do-dia/{fonte}/{DIA}/"
    r = requests.get(url, headers=HEADERS, timeout=15)
    print(f"\n📆 {fonte.upper()} | {DIA} — Status: {r.status_code}")

    if r.status_code == 200:
        payload = r.json()
        dados = payload if isinstance(payload, list) else payload.get("dados", [])
        df = pd.DataFrame(dados)
        print(f"   ✅ {len(df)} jogos | Colunas: {list(df.columns)}")
        display(df.head())
    else:
        print(f"   ⚠️  {r.text[:300]}")

## 4. Download CSV Jogos do Dia — GET `/api/dados/jogos-do-dia/{fonte}/{data}/download/`

In [ ]:
fonte = "bet365"
DIA   = ONTEM  # Usa ontem para garantir que os dados já estão disponíveis

url = f"{BASE_URL}/api/dados/jogos-do-dia/{fonte}/{DIA}/download/"
r = requests.get(url, headers=HEADERS, timeout=15)

print(f"Status: {r.status_code} | Content-Type: {r.headers.get('Content-Type')}")
print(f"URL: {url}")

if r.status_code == 200:
    ct = r.headers.get("Content-Type", "")
    if "csv" in ct or "text/plain" in ct:
        df_dl = pd.read_csv(io.StringIO(r.text))
        print(f"✅ CSV: {len(df_dl)} linhas × {len(df_dl.columns)} colunas")
        display(df_dl.head())
    else:
        payload = r.json()
        dados = payload if isinstance(payload, list) else payload.get("dados", [])
        df_dl = pd.DataFrame(dados)
        print(f"✅ JSON → {len(df_dl)} linhas")
        display(df_dl.head())
else:
    print(f"❌ Erro: {r.text[:500]}")

## 5. Download Base Histórica Completa — GET `/api/dados/{fonte}/download/`

In [ ]:
# ── Funções utilitárias ────────────────────────────────────────────────────────
def drop_reset_index(df):
    df = df.dropna()
    df = df.reset_index(drop=True)
    df.index += 1
    return df

def get_dataframe(fonte, params=None):
    url = f"{BASE_URL}/api/dados/{fonte}/download/"
    print(f"Baixando dados da fonte '{fonte}'...")
    print(f"URL: {url}")
    response = requests.get(url, headers=HEADERS, params=params, timeout=60)
    if response.status_code == 200:
        df = pd.read_csv(io.BytesIO(response.content))
        print(f"✅ Sucesso! {len(df)} linhas × {len(df.columns)} colunas")
        return df
    print(f"❌ Erro: {response.status_code} — {response.text[:300]}")
    return pd.DataFrame()

# ── Bet365 — base completa ─────────────────────────────────────────────────────
FONTE = "bet365"
base  = get_dataframe(FONTE, params={})

if not base.empty:
    df = base.copy()
    df = df.sort_values(by=["Date", "Time"], ascending=[True, True])
    df = drop_reset_index(df)
    print(f"\nColunas: {list(df.columns)}")
    display(df)

In [ ]:
# ── Betfair — base completa ────────────────────────────────────────────────────
FONTE = "betfair"
base  = get_dataframe(FONTE, params={})

if not base.empty:
    df = base.copy()
    df = df.sort_values(by=["Date", "Time"], ascending=[True, True])
    df = drop_reset_index(df)
    print(f"\nColunas: {list(df.columns)}")
    display(df)

In [ ]:
# ── FootyStats — base completa ────────────────────────────────────────────────
FONTE = "footystats"
base  = get_dataframe(FONTE, params={})

if not base.empty:
    df = base.copy()
    df = df.sort_values(by=["Date", "Time"], ascending=[True, True])
    df = drop_reset_index(df)
    print(f"\nColunas: {list(df.columns)}")
    display(df)

## 6. Inspecionar Resposta Bruta (debug)

In [ ]:
# Célula de debug — inspecione qualquer endpoint aqui
fonte = "bet365"
DIA   = ONTEM

url = f"{BASE_URL}/api/dados/jogos-do-dia/{fonte}/{DIA}/"
r = requests.get(url, headers=HEADERS, timeout=15)

print(f"URL:          {r.url}")
print(f"Status:       {r.status_code}")
print(f"Content-Type: {r.headers.get('Content-Type')}")
print(f"\n--- Body (primeiros 2000 chars) ---")
print(r.text[:2000])